# 02｜八列表与远端三状态一致性检查

本 Notebook 不生成信号。它同时读取 01 生成的八列表、包内八列表冻结参考和指定的远端三状态，分别做完整八列逐日对比与三状态逐日对比。

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

package_root = Path(os.environ.get('FINAL_UPLOAD_PACKAGE_ROOT', '/home/hzy/cta/最终冻结运行上传包_含零段反转_20260820_1350')).expanduser()
if not package_root.is_absolute():
    raise ValueError('FINAL_UPLOAD_PACKAGE_ROOT 必须是绝对路径')
package_root = package_root.resolve()
output = Path(os.environ.get('UPLOAD_OUTPUT_DIR', str(package_root / 'runtime_outputs'))).expanduser()
remote = Path(os.environ.get('REMOTE_THREE_STATE_PATH', '/home/hzy/cta/三状态冻结/IC_1545_three_state_and_downside_warning.csv')).expanduser()
for label, path in [('UPLOAD_OUTPUT_DIR', output), ('REMOTE_THREE_STATE_PATH', remote)]:
    if not path.is_absolute():
        raise ValueError(f'{label} 必须是绝对路径')
output = output.resolve()
remote = remote.resolve()
generated = output / '最终执行日简表.csv'
local = package_root / 'expected' / 'local_freeze' / '最终执行日简表_零段反转冻结参考.csv'
command = [sys.executable, str(package_root / 'src' / 'compare_compact_output.py'), '--generated', str(generated), '--local', str(local), '--three-state', str(remote), '--output', str(output)]
subprocess.run(command, cwd=str(package_root), check=True)

In [ ]:
conclusion_path = output / '八列表一致性结论.json'
conclusion = json.loads(conclusion_path.read_text(encoding='utf-8'))
display(conclusion)
assert conclusion['all_dates_and_signals_match']
assert conclusion['all_dates_and_three_state_match']
assert conclusion['success']
print('八列表与远端三状态均逐日一致')